## Map `SOX4` Targets to Transcripts

In [1]:
%%bash

bedtools="/project/PlatigLab/software/bedtools-v2.31.1/bin/bedtools"
ucsc_gene_file='../../../inputs/resource_files/mm10_gencode_vm23_knownGene.bed'
input_file='../3_bedtools_intersect/outputs/MA0867.2.upstream_750.bed.gz'
output_file='all_SOX4_targets_750_upstream.bed'

gunzip -c ${input_file} | grep SOX4 | $bedtools window -a $ucsc_gene_file -b stdin -l 5000 -r 0 -sw > $output_file

gzip $output_file


## Load `SOX4` File & Transcripts-to-Genes Mapping File

In [2]:
import polars as pl
_=pl.Config.set_tbl_rows(1000)



from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [3]:
mapping_file = pl.read_csv("../../../inputs/mouse_transcripts_genes_mapping/mm10_genes_and_transcripts.tsv", separator="\t")
mapping_file.head()

Gene stable ID,Transcript stable ID,Gene name
str,str,str
"""ENSMUSG00000064370""","""ENSMUST00000082421""","""mt-Cytb"""
"""ENSMUSG00000064368""","""ENSMUST00000082419""","""mt-Nd6"""
"""ENSMUSG00000064367""","""ENSMUST00000082418""","""mt-Nd5"""
"""ENSMUSG00000064363""","""ENSMUST00000082414""","""mt-Nd4"""
"""ENSMUSG00000065947""","""ENSMUST00000084013""","""mt-Nd4l"""


In [4]:
sox4_data = pl.read_csv("./all_SOX4_targets_750_upstream.bed.gz", separator="\t", has_header=False)

sox4_data = sox4_data.filter(pl.col('column_17') > 950)
sox4_data = sox4_data.with_columns(
    pl.col("column_4").str.split(".").list.get(0).alias("column_4")
)

sox4_data.shape
sox4_data.head()

(193263, 28)

column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28
str,i64,i64,str,i64,str,i64,i64,i64,i64,str,str,str,i64,i64,str,i64,i64,str,str,i64,i64,str,i64,str,i64,i64,str
"""chr1""",3214481,3671498,"""ENSMUST00000070533""",0,"""-""",3216021,3671348,0,3,"""2487,200,947,""","""0,207220,456070,""","""chr1""",3319573,3319583,"""SOX4""",963,450,"""+""","""chr1""",3319254,3320257,"""mm_5071.1""",1,"""-""",3319505,3319506,"""255,255,0"""
"""chr1""",3214481,3671498,"""ENSMUST00000070533""",0,"""-""",3216021,3671348,0,3,"""2487,200,947,""","""0,207220,456070,""","""chr1""",3320016,3320026,"""SOX4""",960,442,"""-""","""chr1""",3319254,3320257,"""mm_5071.1""",1,"""-""",3319505,3319506,"""255,255,0"""
"""chr1""",3214481,3671498,"""ENSMUST00000070533""",0,"""-""",3216021,3671348,0,3,"""2487,200,947,""","""0,207220,456070,""","""chr1""",3320172,3320182,"""SOX4""",955,430,"""-""","""chr1""",3319254,3320257,"""mm_5071.1""",1,"""-""",3319505,3319506,"""255,255,0"""
"""chr1""",4292980,4409187,"""ENSMUST00000194992""",0,"""-""",4292980,4409187,0,4,"""32,172,636,18,""","""0,58929,59221,116189,""","""chr1""",4409197,4409207,"""SOX4""",985,532,"""-""","""chr1""",4408973,4409990,"""mm_5080.1""",1,"""-""",4409231,4409232,"""255,255,0"""
"""chr1""",4292980,4409187,"""ENSMUST00000194992""",0,"""-""",4292980,4409187,0,4,"""32,172,636,18,""","""0,58929,59221,116189,""","""chr1""",4409197,4409207,"""SOX4""",985,532,"""-""","""chr1""",4408997,4410025,"""mm_5081.1""",1,"""-""",4409261,4409262,"""255,255,0"""


## Filter `SOX4` Results to those Targetting Genes Sunny Cares About

In [5]:
sunny_selected_genes = ["Nr2f1", "Srebf2", "Fads2", "Hipk2", "Ttl", "Rgcc", "Sqle", "Scd2"]
sunny_selected_genes_lower = [gene.lower() for gene in sunny_selected_genes]

sunny_selected_genes_lower

['nr2f1', 'srebf2', 'fads2', 'hipk2', 'ttl', 'rgcc', 'sqle', 'scd2']

In [6]:
mapping_file_subset = mapping_file.filter(
    pl.col("Gene name").str.to_lowercase().is_in(sunny_selected_genes_lower)
)
    
mapping_file_subset


Gene stable ID,Transcript stable ID,Gene name
str,str,str
"""ENSMUSG00000022018""","""ENSMUST00000022595""","""Rgcc"""
"""ENSMUSG00000022463""","""ENSMUST00000023100""","""Srebf2"""
"""ENSMUSG00000022463""","""ENSMUST00000229336""","""Srebf2"""
"""ENSMUSG00000022463""","""ENSMUST00000231065""","""Srebf2"""
"""ENSMUSG00000022463""","""ENSMUST00000230105""","""Srebf2"""
"""ENSMUSG00000022463""","""ENSMUST00000229009""","""Srebf2"""
"""ENSMUSG00000022463""","""ENSMUST00000229518""","""Srebf2"""
"""ENSMUSG00000022463""","""ENSMUST00000230955""","""Srebf2"""
"""ENSMUSG00000022351""","""ENSMUST00000162142""","""Sqle"""


In [7]:
joined = mapping_file_subset.join(
    sox4_data, left_on="Transcript stable ID", right_on="column_4", how="inner"
).sort("Gene name")

for gene in sunny_selected_genes_lower:
    assert joined.filter(pl.col("Gene name").str.to_lowercase() == gene).height > 0, f"No entry found for gene: {gene}"

joined.shape
joined.head()

(106, 30)

Gene stable ID,Transcript stable ID,Gene name,column_1,column_2,column_3,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28
str,str,str,str,i64,i64,i64,str,i64,i64,i64,i64,str,str,str,i64,i64,str,i64,i64,str,str,i64,i64,str,i64,str,i64,i64,str
"""ENSMUSG00000024665""","""ENSMUST00000235937""","""Fads2""","""chr19""",10066572,10088716,0,"""-""",10066572,10088716,0,6,"""40,77,61,36,126,15,""","""0,3699,3991,8544,15992,22129,""","""chr19""",10076846,10076856,"""SOX4""",970,471,"""+""","""chr19""",10076696,10077700,"""mm_164983.1""",1,"""+""",10077448,10077449,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000235937""","""Fads2""","""chr19""",10066572,10088716,0,"""-""",10066572,10088716,0,6,"""40,77,61,36,126,15,""","""0,3699,3991,8544,15992,22129,""","""chr19""",10088921,10088931,"""SOX4""",967,460,"""+""","""chr19""",10088714,10089749,"""mm_152360.1""",1,"""-""",10088981,10088982,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10064064,10064074,"""SOX4""",951,418,"""+""","""chr19""",10063896,10064950,"""mm_152356.1""",1,"""-""",10064173,10064174,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10076846,10076856,"""SOX4""",970,471,"""+""","""chr19""",10076696,10077700,"""mm_164983.1""",1,"""+""",10077448,10077449,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10088921,10088931,"""SOX4""",967,460,"""+""","""chr19""",10088714,10089749,"""mm_152360.1""",1,"""-""",10088981,10088982,"""255,255,0"""


## Rename Columns and Output Data

In [8]:
column_renaming_map = {
    'column_1': 'START OF GENE WINDOW INFO',  
    'column_13': 'START OF MOTIF LOCATION', 
    'column_14': "Motif Start", 
    'column_15': 'Motif End', 
    'column_17': 'Relative Score', 
    'column_20': 'START OF TSS INFO'
    
}

renamed = joined.rename(column_renaming_map)
renamed

Gene stable ID,Transcript stable ID,Gene name,START OF GENE WINDOW INFO,column_2,column_3,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,START OF MOTIF LOCATION,Motif Start,Motif End,column_16,Relative Score,column_18,column_19,START OF TSS INFO,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28
str,str,str,str,i64,i64,i64,str,i64,i64,i64,i64,str,str,str,i64,i64,str,i64,i64,str,str,i64,i64,str,i64,str,i64,i64,str
"""ENSMUSG00000024665""","""ENSMUST00000235937""","""Fads2""","""chr19""",10066572,10088716,0,"""-""",10066572,10088716,0,6,"""40,77,61,36,126,15,""","""0,3699,3991,8544,15992,22129,""","""chr19""",10076846,10076856,"""SOX4""",970,471,"""+""","""chr19""",10076696,10077700,"""mm_164983.1""",1,"""+""",10077448,10077449,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000235937""","""Fads2""","""chr19""",10066572,10088716,0,"""-""",10066572,10088716,0,6,"""40,77,61,36,126,15,""","""0,3699,3991,8544,15992,22129,""","""chr19""",10088921,10088931,"""SOX4""",967,460,"""+""","""chr19""",10088714,10089749,"""mm_152360.1""",1,"""-""",10088981,10088982,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10064064,10064074,"""SOX4""",951,418,"""+""","""chr19""",10063896,10064950,"""mm_152356.1""",1,"""-""",10064173,10064174,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10076846,10076856,"""SOX4""",970,471,"""+""","""chr19""",10076696,10077700,"""mm_164983.1""",1,"""+""",10077448,10077449,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10088921,10088931,"""SOX4""",967,460,"""+""","""chr19""",10088714,10089749,"""mm_152360.1""",1,"""-""",10088981,10088982,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10102174,10102184,"""SOX4""",953,424,"""+""","""chr19""",10101251,10102282,"""mm_152364.1""",1,"""-""",10101516,10101517,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10102174,10102184,"""SOX4""",953,424,"""+""","""chr19""",10101302,10102309,"""mm_152365.1""",1,"""-""",10101555,10101556,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10102174,10102184,"""SOX4""",953,424,"""+""","""chr19""",10101477,10102500,"""mm_152366.1""",1,"""-""",10101738,10101739,"""255,255,0"""
"""ENSMUSG00000024665""","""ENSMUST00000025567""","""Fads2""","""chr19""",10062764,10101746,0,"""-""",10064262,10101429,0,12,"""1550,126,80,97,98,77,61,126,10…","""0,1921,2837,3487,3750,7507,779…","""chr19""",10102174,10102184,"""SOX4""",953,424,"""+""","""chr19""",10101559,10102570,"""mm_152367.1""",1,"""-""",10101814,10101815,"""255,255,0"""


In [9]:
renamed.write_csv('sunny_selected_genes_motif_locations.csv')